Imports

In [16]:
import polars as pl
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import xgboost as xgb
from sklearn.metrics import confusion_matrix, accuracy_score

In [17]:
event_path = r"Data\instance_events-000000000000.json.gz"
usage_path = r"Data\instance_usage-000000000000.json.gz"

In [18]:
# Quick debug to see the actual 2019 Usage schema
usage_sample = pl.read_ndjson(usage_path, n_rows=5)
print(usage_sample.columns)
print(usage_sample.head(1))
event_sample = pl.read_ndjson(event_path, n_rows=5)
print(event_sample.columns)
print(event_sample.head(1))

['start_time', 'end_time', 'collection_id', 'instance_index', 'machine_id', 'alloc_collection_id', 'alloc_instance_index', 'collection_type', 'average_usage', 'maximum_usage', 'random_sample_usage', 'assigned_memory', 'page_cache_memory', 'cycles_per_instruction', 'memory_accesses_per_instruction', 'sample_rate', 'cpu_usage_distribution', 'tail_cpu_usage_distribution']
shape: (1, 18)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ start_tim ┆ end_time  ┆ collectio ┆ instance_ ┆ … ┆ memory_ac ┆ sample_ra ┆ cpu_usage ┆ tail_cpu │
│ e         ┆ ---       ┆ n_id      ┆ index     ┆   ┆ cesses_pe ┆ te        ┆ _distribu ┆ _usage_d │
│ ---       ┆ str       ┆ ---       ┆ ---       ┆   ┆ r_instruc ┆ ---       ┆ tion      ┆ istribut │
│ str       ┆           ┆ str       ┆ str       ┆   ┆ tio…      ┆ f64       ┆ ---       ┆ ion      │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆           ┆ list[f64] ┆ ---      │
│      

In [19]:
def extract_metric(col_name, key):
    extracted = (
        pl.col(col_name)
        .cast(pl.Utf8)
        .str.extract(rf"'{key}':\s*([\d\.eE-]+)", 1)
        .cast(pl.Float64, strict=False)
    )
    
    return pl.coalesce([extracted, pl.col(col_name).cast(pl.Float64, strict=False)]).fill_null(0.0)

In [20]:
def load_master_class_2019(event_path, usage_path, n_rows=5000000):
    event_schema = {
        "machine_id": pl.String,
        "time": pl.String,
        "type": pl.String,
        "priority": pl.String,
        "resource_request": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)])
    }
    
    usage_schema = {
        "machine_id": pl.String,
        "start_time": pl.String,
        "average_usage": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)]),
        "maximum_usage": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)]),
        "assigned_memory": pl.Float64,
        "cycles_per_instruction": pl.Float64
    }

    events = (
        pl.scan_ndjson(event_path, n_rows=n_rows, schema=event_schema)
        .select([
            pl.col("machine_id").cast(pl.Float64),
            pl.col("time").cast(pl.Float64),
            pl.col("type").cast(pl.Int32).is_in([2, 3, 4]).cast(pl.Int8).alias("error_signal"),
            pl.col("priority").cast(pl.Int32).alias("priority_level"),
            pl.col("resource_request").struct.field("cpus").alias("demand_cpu")
        ])
        .sort("time")
    )

    usage = (
        pl.scan_ndjson(usage_path, n_rows=n_rows, schema=usage_schema)
        .select([
            pl.col("machine_id").cast(pl.Float64),
            pl.col("start_time").cast(pl.Float64).alias("time"),
            pl.col("average_usage").struct.field("cpus").alias("utilization_cpu"),
            pl.col("maximum_usage").struct.field("cpus").alias("saturation_cpu"),
            pl.col("assigned_memory").alias("utilization_mem"),
            pl.col("cycles_per_instruction").alias("delay_cpi")
        ])
        .sort("time")
    )

    return events.join_asof(
        usage, 
        on="time", 
        by="machine_id", 
        strategy="backward"
    ).collect()

df_final = load_master_class_2019(event_path, usage_path)
print(f"Joined Dataset Shape: {df_final.shape}")
df_final

C:\Users\sabbu\AppData\Local\Temp\ipykernel_26736\2215271374.py:49: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  ).collect()


Joined Dataset Shape: (5000000, 9)


machine_id,time,error_signal,priority_level,demand_cpu,utilization_cpu,saturation_cpu,utilization_mem,delay_cpi
f64,f64,i8,i32,f64,f64,f64,f64,f64
null,0.0,0,500,0.096558,null,null,null,null
null,0.0,0,500,0.087769,null,null,null,null
null,0.0,1,500,0.096558,null,null,null,null
null,0.0,1,500,0.087769,null,null,null,null
5.3772e10,0.0,1,500,0.096558,null,null,null,null
…,…,…,…,…,…,…,…,…
null,9.2234e18,0,0,null,null,null,null,null
null,9.2234e18,0,0,null,null,null,null,null
null,9.2234e18,0,0,null,null,null,null,null


In [21]:
def finalize_ml_data(df, window_size=5):
    feature_cols = [
        "demand_cpu", "utilization_cpu", "saturation_cpu", 
        "utilization_mem", "delay_cpi", "priority_level"
    ]
    
    scaler = StandardScaler()
    df_clean = df.fill_null(0.0)
    scaled_data = scaler.fit_transform(df_clean.select(feature_cols).to_numpy())
    
    df_scaled = df_clean.with_columns([
        pl.Series(name=feature_cols[i], values=scaled_data[:, i]) 
        for i in range(len(feature_cols))
    ])

    X, y = [], []
    for _, group in df_scaled.group_by("machine_id"):
        if len(group) <= window_size:
            continue
        
        feat_arr = group.select(feature_cols).to_numpy().astype('float32')
        label_arr = group.select("error_signal").to_numpy().astype('float32')

        for i in range(len(group) - window_size):
            X.append(feat_arr[i : i + window_size])
            y.append(label_arr[i + window_size])
            
    return np.array(X), np.array(y)

X, y = finalize_ml_data(df_final)

print(f"Feature Tensor (X) Shape: {X.shape}")
print(f"Label Array (y) Shape: {y.shape}")

Feature Tensor (X) Shape: (4950077, 5, 6)
Label Array (y) Shape: (4950077, 1)


Test bench Models

In [22]:
def build_CNN_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64, kernel_size=2, activation='relu'),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_LSTM_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_Hybrid_model(input_shape):
    inputs = Input(shape=input_shape)
    # adding 1d CNN
    x = layers.Conv1D(64, 2, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    #adding LSTM
    x = layers.LSTM(64, return_sequences=True)(x)
    #adding attention
    query_val_attention = layers.Attention()([x,x])
    
    x = layers.GlobalAveragePooling1D()(query_val_attention)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

In [38]:
def print_detailed_analysis(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total = len(y_true)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    false_discovery_rate = fp / (fp + tp) if (fp + tp) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    print(f"\n" + "="*40)
    print(f"{name.upper()} PERFORMANCE REPORT")
    print("="*40)
    
    print(f"DETECTIONS")
    print(f"   True Positives (Caught Failures):  {tp:<8} ({(tp/total)*100:>5.2f}%)")
    print(f"   True Negatives (Correctly Healthy): {tn:<8} ({(tn/total)*100:>5.2f}%)")
    
    print(f"\nERRORS")
    print(f"   False Positives (False Alarms):    {fp:<8} ({(fp/total)*100:>5.2f}%)")
    print(f"   False Negatives (Missed Failures): {fn:<8} ({(fn/total)*100:>5.2f}%)")
    
    print(f"\nCORE METRICS")
    print(f"   F1 Score:  {f1:.4f}  (Balanced Accuracy)")
    print(f"   Precision: {precision:.4f}  (Reliability of Alarms)")
    print(f"   Recall:    {recall:.4f}  (Ability to Catch Failures)")
    print(f"   Specificity: {specificity:.4f}  (Baseline Stability)")
    
    print(f"\nACTIONABLE INSIGHTS")
    print(f"   False Discovery Rate: {false_discovery_rate:.2%}")
    print(f"   Insight: For every 100 alarms, {false_discovery_rate*100:.1f} are likely 'glitches'.")
    print("="*40)

In [28]:
def calculate_lead_time(y_true, y_pred, time_interval_mins=5):
    """
    Calculates how much warning (in minutes) the model gives.
    Assumes y_true and y_pred are sequences for a single machine.
    """
    lead_times = []
    
    # We look for the first '1' in predictions that leads to a '1' in truth
    for i in range(len(y_pred) - 1):
        if y_pred[i] == 1 and y_true[i] == 0:
            # Look ahead to see when the actual failure happens
            for look_ahead in range(i + 1, len(y_true)):
                if y_true[look_ahead] == 1:
                    # Lead time = steps between prediction and reality * 5 mins
                    time_diff = (look_ahead - i) * time_interval_mins
                    lead_times.append(time_diff)
                    break 
    
    if not lead_times:
        return 0
        
    return np.mean(lead_times)

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

In [25]:
def test_industry_baselines(X_test, y_test):    
    # Static Threshold
    y_pred_static = (X_test[:, -1, 2] > 0.8) | (X_test[:, -1, 4] > 0.8)
    f1_static = f1_score(y_test, y_pred_static.astype(int))
    
    # EWMA Baseline
    alpha = 0.3
    ewma_val = np.zeros(X_test.shape[0])
    for t in range(X_test.shape[1]):
        ewma_val = alpha * X_test[:, t, 2] + (1 - alpha) * ewma_val
    
    y_pred_ewma = (ewma_val > 0.7)
    f1_ewma = f1_score(y_test, y_pred_ewma.astype(int))
    
    print(f"Static Threshold F1: {f1_static:.4f}")
    print(f"EWMA Moving Average F1: {f1_ewma:.4f}")

test_industry_baselines(X_test, y_test)

Static Threshold F1: 0.2534
EWMA Moving Average F1: 0.0452


In [ ]:
models_to_test = {
    "Isolation Forest": "baseline_unsupervised",
    "Logistic Reg": "baseline_supervised",
    "XGBoost": "baseline_supervised",
    "Individual CNN": build_CNN_model,
    "Individual LSTM": build_LSTM_model,
    "Hybrid": build_Hybrid_model
}

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

results = {}

for name, factory in models_to_test.items():
    print(f"\n--- Training {name} ---")
    if name == "Isolation Forest":
        iso = IsolationForest(contamination=0.05, max_samples=100000, random_state=42).fit(X_train_flat)
        y_pred = [1 if val == -1 else 0 for val in iso.predict(X_test_flat)]
    
    elif name == "Logistic Reg":
        lr = LogisticRegression(max_iter=1000, solver='saga').fit(X_train_flat, y_train.ravel())
        y_pred = lr.predict(X_test_flat)

    elif name == "XGBoost":
        xgb_model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            tree_method='hist',
            random_state=42
        )
        xgb_model.fit(X_train_flat, y_train.ravel())
        y_pred = xgb_model.predict(X_test_flat)
        
    else:
        m = factory(X_train.shape[1:])
        m.fit(
            X_train, y_train, 
            epochs=10, 
            batch_size=512, 
            verbose=1, 
            validation_split=0.1,
            callbacks=[early_stop]
        )
        y_pred = (m.predict(X_test) > 0.5).astype(int)
    
    score = f1_score(y_test, y_pred)
    results[name] = score
    print(f"{name} F1 Score on Test Set: {score:.4f}")

print("\n" + "="*40)
print(f"PHASE 2 WINNER: {max(results, key=results.get)}")
print("="*40)


--- Training Isolation Forest ---
Isolation Forest F1 Score on Test Set: 0.1149

--- Training Logistic Reg ---
Logistic Reg F1 Score on Test Set: 0.3826

--- Training XGBoost ---
XGBoost F1 Score on Test Set: 0.6848

--- Training Individual CNN ---
Epoch 1/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - AUC: 0.7752 - loss: 0.5608 - val_AUC: 0.8082 - val_loss: 0.5278
Epoch 2/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - AUC: 0.8104 - loss: 0.5239 - val_AUC: 0.8205 - val_loss: 0.5131
Epoch 3/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - AUC: 0.8193 - loss: 0.5131 - val_AUC: 0.8277 - val_loss: 0.5029
Epoch 4/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - AUC: 0.8239 - loss: 0.5076 - val_AUC: 0.8334 - val_loss: 0.4973
Epoch 5/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - AUC: 0.8271 - loss: 0.5036 - val_AUC: 0.8322 - val_loss: 0.4966
Epoch 6/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - AUC: 0.8291 - loss: 0.5008 - val_AUC: 0.8346 - val_loss: 0.4937
Epoch 7/10
6962/6962 ━━━

In [31]:
extended_early_stop = EarlyStopping( monitor='val_loss', patience=5, restore_best_weights=True)

all_predictions = {}
optimized_results = {}

top_models = {
    "Individual CNN": build_CNN_model,
    "Hybrid": build_Hybrid_model
}

for name, factory in top_models.items():
    print(f"\n--- Deep Training {name} ---")
    m = factory(X_train.shape[1:])
    
    history = m.fit(
        X_train, y_train, 
        epochs=50,           
        batch_size=512,
        verbose=1, 
        validation_split=0.1,
        callbacks=[extended_early_stop]
    )
    
    preds = (m.predict(X_test) > 0.5).astype(int)
    all_predictions[name] = preds

    score = f1_score(y_test, preds)
    optimized_results[name] = score
    print(f"{name} Final F1: {score:.4f}")


--- Deep Training Individual CNN ---
Epoch 1/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - AUC: 0.7777 - loss: 0.5587 - val_AUC: 0.8093 - val_loss: 0.5259
Epoch 2/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - AUC: 0.8113 - loss: 0.5230 - val_AUC: 0.8222 - val_loss: 0.5098
Epoch 3/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - AUC: 0.8198 - loss: 0.5125 - val_AUC: 0.8278 - val_loss: 0.5028
Epoch 4/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - AUC: 0.8244 - loss: 0.5067 - val_AUC: 0.8309 - val_loss: 0.4981
Epoch 5/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - AUC: 0.8272 - loss: 0.5030 - val_AUC: 0.8336 - val_loss: 0.4951
Epoch 6/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - AUC: 0.8294 - loss: 0.5003 - val_AUC: 0.8352 - val_loss: 0.4934
Epoch 7/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - AUC: 0.8312 - loss: 0.4980 - val_AUC: 0.8357 - val_loss: 0.4922
Epoch 8/50
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - AUC: 0.8325 - loss: 0.4964 - val_AUC: 0.8202 - val_loss: 0

In [39]:
print_detailed_analysis("Individual CNN", y_test, all_predictions["Individual CNN"])
print_detailed_analysis("Hybrid", y_test, all_predictions["Hybrid"])


INDIVIDUAL CNN PERFORMANCE REPORT
DETECTIONS
   True Positives (Caught Failures):  304442   (30.75%)
   True Negatives (Correctly Healthy): 446107   (45.06%)

ERRORS
   False Positives (False Alarms):    109549   (11.07%)
   False Negatives (Missed Failures): 129918   (13.12%)

CORE METRICS
   F1 Score:  0.7177  (Balanced Accuracy)
   Precision: 0.7354  (Reliability of Alarms)
   Recall:    0.7009  (Ability to Catch Failures)
   Specificity: 0.8028  (Baseline Stability)

ACTIONABLE INSIGHTS
   False Discovery Rate: 26.46%
   Insight: For every 100 alarms, 26.5 are likely 'glitches'.

HYBRID PERFORMANCE REPORT
DETECTIONS
   True Positives (Caught Failures):  290661   (29.36%)
   True Negatives (Correctly Healthy): 473751   (47.85%)

ERRORS
   False Positives (False Alarms):    81905    ( 8.27%)
   False Negatives (Missed Failures): 143699   (14.51%)

CORE METRICS
   F1 Score:  0.7204  (Balanced Accuracy)
   Precision: 0.7802  (Reliability of Alarms)
   Recall:    0.6692  (Ability to Ca

In [36]:
avg_lead_hybrid = calculate_lead_time(y_test.ravel(), all_predictions["Hybrid"].ravel())
print(f"Average Lead-Time Warning(Hybrid): {avg_lead_hybrid:.2f} minutes")

avg_lead_cnn = calculate_lead_time(y_test.ravel(), all_predictions["Individual CNN"].ravel())
print(f"Average Lead-Time Warning(Individual CNN): {avg_lead_cnn:.2f} minutes")

Average Lead-Time Warning(Hybrid): 11.40 minutes
Average Lead-Time Warning(Individual CNN): 11.41 minutes
